In [1]:
import numpy as np
import matplotlib.pyplot as plt
import time
import sys
import robotic as ry
from raiSimulationEnvATKDEF import RobotSimEnv

/home/said/robotics/lib/python3.9/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


**Visualization tools for Attack Paths**

In [2]:
env = RobotSimEnv(render=True)

/home/said/robotics/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/said/robotics/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(


In [3]:
import os

attack_files = [f for f in os.listdir('attackPaths') if f.startswith('p') ]
for i in range(10):
    env.initializeSimulation()
    random_idx = np.random.randint(0,len(attack_files))
    attack_path = np.load('attackPaths/'+attack_files[random_idx])
    env.followSplinePath(attack_path,0.3,0.01)


-- kin_physx.cpp:addJoint:298(0) ADDING JOINT l_panda_joint7-sword_0 of type rigid with rel [0, 0, 0]
-- kin_physx.cpp:addJoint:298(0) ADDING JOINT r_panda_joint7-shi of type rigid with rel [0, 0, 0]


KeyboardInterrupt: 

**Visualization tools for Defence Paths**

In [4]:
env = RobotSimEnv(render=True)

In [5]:
import os

attack_files = os.listdir('defence paths')
for i in range(100):
    env.initializeSimulation()
    random_idx = np.random.randint(0,len(attack_files))
    attack_path = np.load('defence paths/'+attack_files[random_idx])
    if len(attack_path.shape) == 4:
        attack_path = attack_path[0]
    env.followSplinePath(attack_path[:,0,:],0.5,0.01)

-- kin_physx.cpp:addJoint:298(0) ADDING JOINT l_panda_joint7-sword_0 of type rigid with rel [0, 0, 0]
-- kin_physx.cpp:addJoint:298(0) ADDING JOINT r_panda_joint7-shi of type rigid with rel [0, 0, 0]


KeyboardInterrupt: 

Visualise First Training Results

In [6]:
import stable_baselines3
from raiSimulationEnvATKDEF import RobotSimEnv
import robotic as ry
import numpy as np
import time

import os
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback, CallbackList
from stable_baselines3 import PPO
import gymnasium as gym

class RewardLoggerCallback(BaseCallback):
    def __init__(self, log_file: str, verbose: int = 0):
        super(RewardLoggerCallback, self).__init__(verbose)
        self.log_file = log_file
        self.episode_rewards = []
        self.current_episode_reward = 0

        # Create the log file if it doesn't exist
        if not os.path.exists(self.log_file):
            with open(self.log_file, 'w') as f:
                f.write("Episode,Total Reward\n")

    def _on_step(self) -> bool:
        # Check if the episode has ended by using `done`
        dones = self.locals["dones"]
        rewards = self.locals["rewards"]

        # Accumulate rewards for the current episode
        self.current_episode_reward += rewards[0]

        # If the episode is done, log the reward
        if dones[0]:
            self.episode_rewards.append(self.current_episode_reward)
            with open(self.log_file, 'a') as f:
                f.write(f"{len(self.episode_rewards)},{self.current_episode_reward}\n")
            # Reset the reward counter for the next episode
            self.current_episode_reward = 0


        return True

    def _on_training_end(self) -> None:
        # Optionally summarize results at the end of training
        print("Training finished. Total episodes:", len(self.episode_rewards))
        print("Episode rewards:", self.episode_rewards)

class CustomCheckpointCallback(BaseCallback):
    def __init__(self, save_freq, save_path, verbose=0):
        super(CustomCheckpointCallback, self).__init__(verbose)
        self.save_freq = save_freq
        self.save_path = save_path

    def _on_step(self) -> bool:
        # Save the model every `save_freq` steps
        if self.n_calls % self.save_freq == 0:
            model_path = f"{self.save_path}/model_checkpoint_{self.n_calls}_steps.zip"
            self.model.save(model_path)
            if self.verbose > 0:
                print(f"Model saved at step {self.n_calls} to {model_path}")
        return True

class StaticOpponentWrapper(gym.Wrapper):
    """
    A wrapper for a 1v1 environment where one agent is controlled by a fixed, pre-trained policy.
    """
    def __init__(self, env, static_policy_attacker, static_policy_defender,attacker=True,test=False):
        super(StaticOpponentWrapper, self).__init__(env)
        self.static_policy_attacker = static_policy_attacker
        self.static_policy_defender = static_policy_defender
        self.attacker = attacker
        self.test = test

    def step(self, action):
        # Get the static opponent's action
        obs = self.env.state  # Modify this based on how your env works
        attacker_action, _ = self.static_policy_attacker.predict(obs, deterministic=True)
        defender_action, _ = self.static_policy_defender.predict(obs, deterministic=True)
        # Combine both actions into a joint action
        if self.attacker:
            joint_action = (attacker_action, action)
        else:
            joint_action = (action,defender_action)
        
        if self.test:
            joint_action = (attacker_action, defender_action)

        # Step the environment with both actions
        obs, reward, done, truncated, info = self.env.step(joint_action)
        return obs, reward, done, truncated, info

    def reset(self, **kwargs):
        return self.env.reset(**kwargs)

from collections import OrderedDict

def update_ordered_dict_keys(ordered_dict, key_mapping):
    """
    Update keys in an OrderedDict based on a key mapping dictionary.

    Args:
        ordered_dict (OrderedDict): The OrderedDict to update.
        key_mapping (dict): A dictionary where keys are the old keys to be replaced,
                            and values are the new keys.

    Returns:
        OrderedDict: A new OrderedDict with updated keys.
    """
    updated_dict = OrderedDict()

    for old_key, value in ordered_dict.items():
        # Use the new key if it exists in the key_mapping, otherwise keep the old key
        new_key = key_mapping.get(old_key, old_key)
        updated_dict[new_key] = value

    return updated_dict

def delete_values_from_ordered_dict(ordered_dict, keys_to_delete):
    """
    Create a new OrderedDict with specific keys removed.

    Args:
        ordered_dict (OrderedDict): The original OrderedDict.
        keys_to_delete (list): A list of keys to be removed.

    Returns:
        OrderedDict: A new OrderedDict with specified keys removed.
    """
    return OrderedDict((key, value) for key, value in ordered_dict.items() if key not in keys_to_delete)


def select_keys(ordered_dict, keys):
    """
    Select specific keys from an OrderedDict and return a new OrderedDict with only the selected keys.

    Parameters:
        ordered_dict (OrderedDict): The original OrderedDict.
        keys (list): List of keys to select from the OrderedDict.

    Returns:
        OrderedDict: A new OrderedDict containing only the selected keys.
    """
    return OrderedDict((key, ordered_dict[key]) for key in keys if key in ordered_dict)


In [7]:
from stable_baselines3 import PPO
import time
import torch

env = RobotSimEnv(render_mode='human',staticAttacker=True, staticDefender=True)
policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))

attackModel = PPO.load("sword_best")
defenceModel = PPO.load("shield_best")

wrapped_env = StaticOpponentWrapper(env, attackModel,defenceModel,attacker=True,test=True)

policy_kwargs = dict(net_arch=dict(pi=[256, 256], vf=[256, 256]))


model = PPO("MlpPolicy", wrapped_env, policy_kwargs=policy_kwargs, verbose=0, device='cpu')   

vec_env = model.get_env()
obs = vec_env.reset()
time.sleep(1)

attackSuccessesRates = []
defenceSuccessesRates = []
for i in range(1000):
    action, _states = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    if done:
        print("attack success rate", env.attackSuccess/(env.attackSuccess+env.defenceSuccess))
        print("defence success rate", env.defenceSuccess/(env.attackSuccess+env.defenceSuccess))
    env.render()


/home/said/robotics/lib/python3.9/site-packages/gymnasium/spaces/box.py:235: UserWarning: WARN: Box low's precision lowered by casting to float32, current low.dtype=float64
  gym.logger.warn(
/home/said/robotics/lib/python3.9/site-packages/gymnasium/spaces/box.py:305: UserWarning: WARN: Box high's precision lowered by casting to float32, current high.dtype=float64
  gym.logger.warn(
/home/said/robotics/lib/python3.9/site-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(


-- kin_physx.cpp:addJoint:298(0) ADDING JOINT l_panda_joint7-sword_0 of type rigid with rel [0, 0, 0]
-- kin_physx.cpp:addJoint:298(0) ADDING JOINT r_panda_joint7-shi of type rigid with rel [0, 0, 0]
Success ('sword_1', 'r_panda_coll5', -0.036841034856630615)
-- kin_physx.cpp:addJoint:298(0) ADDING JOINT l_panda_joint7-sword_0 of type rigid with rel [0, 0, 0]
-- kin_physx.cpp:addJoint:298(0) ADDING JOINT r_panda_joint7-shi of type rigid with rel [0, 0, 0]


AttributeError: 'RobotSimEnv' object has no attribute 'attackSuccess'